# Meta L6 AI-Native DE On-Site Mock - Working Code + Theory

**Purpose:** Hands-on reference for the 5 on-site rounds. Each cell is runnable.

**Rounds:**
1. Greet (15m) - pitch
2. AI-Native DE Full Stack (60m) - modeling, RAG design, MMR code, eval SQL
3. Sr Tech Design (60m) - PYMK feed with anti-hallucination
4. Full Stack (60m) - trace reviewer tool
5. Sr Ownership (30m) - behavioral

**L6 cheat sheet keywords to drop in every answer:**
`end-to-end` . `versioned prompts` . `grounded generation` . `eval dataset` . `shadow index` . `bounded fanout` . `idempotent` . `SLO` . `LLM-as-judge`


---

## ROUND 2 - Q1: Data Modeling for AI-Native RAG

**What they test:** Can you design a production schema for a RAG system with GDPR, chunk versioning, and feedback loops?

**Why this schema:**
- `documents` is source-of-truth, with `is_deleted` tombstone (GDPR requires soft delete + async purge, NOT hard delete from `traces` for 30 days)
- `chunks` is the actual data model - `(doc_id, chunk_hash, embedding_model_version)` UNIQUE prevents duplicate work when re-embedding
- `embeddings` is SEPARATE - vector(1536) = 6KB/row, expensive to scan; split lets you run analytics on `chunks` without loading vectors
- `prompts` is versioned like code - NEVER edit in place (you lose the ability to A/B test or roll back)
- `traces` is a FACT TABLE - partition by day, 1B rows/day at Meta scale
- `feedback` is the gold dataset - `corrected_answer IS NOT NULL` = labeled eval data


In [ ]:
schema_sql = '''
-- 1. Source of truth (with GDPR tombstone)
CREATE TABLE documents (
    doc_id        UUID PRIMARY KEY,
    source        VARCHAR(50),
    uri           TEXT,
    owner_id      UUID,
    chunk_strategy VARCHAR(50),
    ingested_at   TIMESTAMP,
    is_deleted    BOOLEAN DEFAULT FALSE,
    hash          VARCHAR(64)
);

-- 2. Chunks = your data model
CREATE TABLE chunks (
    chunk_id                UUID PRIMARY KEY,
    doc_id                  UUID REFERENCES documents(doc_id),
    chunk_index             INT,
    text                    TEXT,
    token_count             INT,
    embedding_model_version VARCHAR(50),
    chunk_hash              VARCHAR(64),
    created_at              TIMESTAMP,
    UNIQUE (doc_id, chunk_hash, embedding_model_version),
    INDEX idx_model_version (embedding_model_version)
);

-- 3. Embeddings SEPARATE (vector column = 6KB/row)
CREATE TABLE embeddings (
    chunk_id      UUID PRIMARY KEY REFERENCES chunks(chunk_id),
    embedding     vector(1536),
    model_version VARCHAR(50),
    created_at    TIMESTAMP
);

-- 4. Versioned prompts (NEVER edit in place)
CREATE TABLE prompts (
    prompt_id       UUID PRIMARY KEY,
    prompt_template TEXT,
    version         INT,
    created_at      TIMESTAMP
);

-- 5. Traces = fact table, 1B rows/day at Meta scale
CREATE TABLE traces (
    trace_id            UUID,
    user_id             UUID,
    query               TEXT,
    retrieved_chunk_ids UUID[],
    prompt_version      INT REFERENCES prompts(version),
    response            TEXT,
    latency_ms          INT,
    cost_tokens         INT,
    created_at          TIMESTAMP,
    PRIMARY KEY (trace_id, created_at)
) PARTITION BY RANGE (created_at);

-- 6. Feedback = gold dataset for offline eval
CREATE TABLE feedback (
    feedback_id      UUID PRIMARY KEY,
    trace_id         UUID,
    rating           TEXT CHECK (rating IN ('thumbs_up', 'thumbs_down')),
    corrected_answer TEXT,
    created_at       TIMESTAMP
);
'''
print(schema_sql)


**L6 talking points to say out loud while drawing this:**
1. "Chunking IS a modeling decision - different docs need different splitters. I store `chunk_strategy` on `documents`."
2. "GDPR: I tombstone `documents.is_deleted=TRUE`, then an async purge job deletes from `embeddings` and calls vector DB delete. I keep `traces` for 30 days for abuse investigation."
3. "Embeddings as separate table because `vector(1536)` is 6KB/row. If I store them inline with chunks, every analytics query loads the full vector."
4. "Prompts are versioned like code. If I edit prompt_v3 in place, I lose the ability to A/B test or roll back when satisfaction drops."
5. "Traces is partitioned by day - at Meta scale, 1B rows/day, you cannot query unpartitioned. Drop old partitions after 30 days, not DELETE rows."
6. "Feedback table is the eval set. `SELECT * FROM feedback WHERE corrected_answer IS NOT NULL` is labeled data I replay new prompts against before shipping."


---

## ROUND 2 - Q2: RAG Pipeline Design (10M docs, <500ms p95, 5min freshness)

**What they test:** Can you design for scale, freshness, AND cost?

**Why this architecture (Lambda for RAG):**
- **Online path** (freshness): Kafka + Flink CDC -> invalidate Qdrant ID + upsert new chunks. Full reindex is too slow for 10M docs.
- **Offline path** (scale): Spark batch for backfill at 5k RPS - re-embed 10M docs in ~30 min
- **Serving path** (latency): hybrid search (BM25 + vector) -> reranker (cross-encoder) -> Top 8 -> LLM. BM25 catches exact-match (names, IDs) that vector misses.
- **Cost control**: skip embedding for low-quality docs (>5k tokens, low info score). 30% of repeat queries -> cache query embeddings in Redis.


**L6 phrases to drop:**
- "Lambda for RAG: offline batch backfill + online incremental freshness = best of both"
- "Hybrid search: BM25 + vector. Vector misses exact matches (employee IDs, product SKUs). Reranker cross-encoder is in the serving path, not the indexing path - too slow for indexing."
- "Cache query embeddings in Redis - repeat queries are 30% of traffic at Meta scale, saves embedding cost"
- "Skip embedding for low-quality docs (>5k tokens, low info score via TF-IDF heuristic) - cost control without hurting quality"
- "Idempotent via chunk_hash - if hash unchanged, skip re-embed. Versioning via embedding_model_version - old chunks stay valid until you re-embed them."


---

## ROUND 2 - Q3: MMR Rerank Coding (the actual whiteboard question)

**What they test:** Can you implement a real algo under pressure, not just describe it?

**Why MMR:** Standard vector search returns redundant chunks (e.g., 5 chunks from the same section). MMR = Maximal Marginal Relevance - balances **relevance** vs **diversity** so you get 8 chunks that cover different aspects, not duplicates of the same one.

**The formula:**
```
MMR(d) = LAMBDA * sim(d, query) - (1-LAMBDA) * max[sim(d, d_selected)]
         ^                          ^
     relevance               diversity penalty
```
- LAMBDA=0.7 -> favor relevance (default)
- LAMBDA=0.3 -> favor diversity (when chunks are very similar)


In [ ]:
import numpy as np

def mmr_rerank(query_emb, chunk_embs, chunks, lambda_=0.7, k=8):
    """
    Maximal Marginal Relevance reranking.

    Args:
        query_emb:   [D]            - query embedding vector
        chunk_embs:  [N, D]         - all candidate chunk embeddings
        chunks:      list of N      - the actual chunk texts/metadata
        lambda_:     float          - relevance vs diversity tradeoff (0.7 = favor relevance)
        k:           int            - number of chunks to return

    Returns:
        list of k chunks that are relevant AND diverse

    Complexity: O(k * N * D) - fine for k=8, N=50.
    For N=10k+, use approximate MMR via HNSW.
    """
    selected = []
    candidates = list(range(len(chunks)))

    # Pre-compute cosine similarity of each chunk to query (assumes normalized vectors)
    sim_to_query = chunk_embs @ query_emb  # [N]

    while len(selected) < k and candidates:
        if not selected:
            # First pick: most relevant to query
            best = max(candidates, key=lambda i: sim_to_query[i])
        else:
            # Score each candidate: relevance - diversity penalty
            scores = []
            for i in candidates:
                # max similarity to ANY already-selected chunk
                max_sim_to_selected = max(
                    chunk_embs[i] @ chunk_embs[j]
                    for j in selected
                )
                mmr_score = (lambda_ * sim_to_query[i]
                           - (1 - lambda_) * max_sim_to_selected)
                scores.append((mmr_score, i))
            best = max(scores)[1]

        selected.append(best)
        candidates.remove(best)

    return [chunks[i] for i in selected]


# TEST IT
np.random.seed(42)
N, D = 10, 64
query_emb = np.random.randn(D)
query_emb /= np.linalg.norm(query_emb)  # normalize for cosine sim

chunk_embs = np.random.randn(N, D)
chunk_embs /= np.linalg.norm(chunk_embs, axis=1, keepdims=True)  # normalize

chunks = [f"chunk_{i}: this is some text about topic {i}" for i in range(N)]

result = mmr_rerank(query_emb, chunk_embs, chunks, lambda_=0.7, k=3)
print("Top 3 MMR-selected chunks:")
for i, c in enumerate(result):
    print(f"  {i+1}. {c}")
print()
print("Without MMR, you'd just take top-3 by cosine sim to query -")
print("which often returns 3 near-duplicates. MMR forces diversity.")


In [ ]:
def pure_relevance_top_k(query_emb, chunk_embs, chunks, k=3):
    """Baseline: just take top-k by cosine sim, no diversity."""
    sims = chunk_embs @ query_emb
    top_k_idx = np.argsort(-sims)[:k]
    return [chunks[i] for i in top_k_idx], sims[top_k_idx]

# Create a scenario where 3 chunks are very similar to each other (duplicates)
np.random.seed(7)
query_emb = np.random.randn(D)
query_emb /= np.linalg.norm(query_emb)

chunk_embs = np.random.randn(N, D)
chunk_embs /= np.linalg.norm(chunk_embs, axis=1, keepdims=True)

# Make chunks 0, 1, 2 nearly identical (simulate duplicates)
chunk_embs[1] = chunk_embs[0] + 0.01 * np.random.randn(D)
chunk_embs[2] = chunk_embs[0] + 0.01 * np.random.randn(D)
chunk_embs[1] /= np.linalg.norm(chunk_embs[1])
chunk_embs[2] /= np.linalg.norm(chunk_embs[2])

chunks = [f"chunk_{i}: content variant {i}" for i in range(N)]

print("=" * 60)
print("PURE RELEVANCE (no diversity penalty) - picks duplicates:")
print("=" * 60)
top_k_relevant, sims = pure_relevance_top_k(query_emb, chunk_embs, chunks, k=3)
for c, s in zip(top_k_relevant, sims):
    print(f"  {c}  (sim={s:.3f})")

print()
print("=" * 60)
print("MMR (lambda=0.7) - picks diverse chunks:")
print("=" * 60)
top_k_mmr = mmr_rerank(query_emb, chunk_embs, chunks, lambda_=0.7, k=3)
for c in top_k_mmr:
    print(f"  {c}")


**L6 line to say while coding:**

> "In prod, I would never code this from scratch - Qdrant has built-in MMR, or I would use Cohere Rerank API. This is the algo behind it though, and interviewers want to see I understand the math. Complexity is O(k*N*D), fine for k=8, N=50. If N=10k+, I would use approximate MMR via HNSW."


---

## ROUND 2 - Q4: SQL - Measure RAG Quality

**What they test:** Can you write analytics SQL on the schema you just designed? Do you know which KPIs matter for RAG?

**The 3 RAG KPIs:**
1. **Satisfaction** = thumbs_up / total (with NULL feedback excluded)
2. **p95 latency** = PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY latency_ms)
3. **Cost** = SUM(cost_tokens) per day (every LLM call has $$ cost)

**Why GROUP BY day, not user:** Aggregating by day gives you a time series for dashboards (Grafana). Per-user is for cohort analysis, not daily monitoring.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

cur.executescript("""
CREATE TABLE traces (
    trace_id TEXT PRIMARY KEY,
    user_id TEXT,
    query TEXT,
    prompt_version INT,
    response TEXT,
    latency_ms INT,
    cost_tokens INT,
    created_at TIMESTAMP
);
CREATE TABLE feedback (
    feedback_id TEXT PRIMARY KEY,
    trace_id TEXT,
    rating TEXT,
    corrected_answer TEXT,
    created_at TIMESTAMP
);
""")

# Seed synthetic data - 7 days of traces, ~100/day
np.random.seed(42)
trace_rows = []
feedback_rows = []
for day_offset in range(7):
    for i in range(100):
        trace_id = f"t_{day_offset}_{i}"
        trace_rows.append((
            trace_id,
            f"u_{i % 20}",
            f"query {i}",
            int(np.random.choice([1, 2, 3])),
            f"response {i}",
            int(np.random.normal(450, 100)),
            int(np.random.normal(800, 200)),
            f"2026-10-{7+day_offset:02d} 12:00:00",
        ))
        if np.random.rand() < 0.3:
            feedback_rows.append((
                f"f_{trace_id}",
                trace_id,
                'thumbs_up' if np.random.rand() < 0.75 else 'thumbs_down',
                None if np.random.rand() < 0.5 else f"corrected_{i}",
                f"2026-10-{7+day_offset:02d} 12:05:00",
            ))

cur.executemany("INSERT INTO traces VALUES (?,?,?,?,?,?,?,?)", trace_rows)
cur.executemany("INSERT INTO feedback VALUES (?,?,?,?,?)", feedback_rows)
conn.commit()

print(f"Seeded {len(trace_rows)} traces and {len(feedback_rows)} feedback rows")


In [ ]:
# Q4 SQL answer: measure RAG quality from traces + feedback
eval_sql = """
SELECT
    DATE(t.created_at) AS day,
    COUNT(*) AS total_traces,
    AVG(CASE WHEN f.rating = 'thumbs_up' THEN 1.0 ELSE 0.0 END) AS satisfaction,
    AVG(t.latency_ms) AS avg_latency_ms,
    SUM(t.cost_tokens) AS total_cost_tokens
FROM traces t
LEFT JOIN feedback f ON t.trace_id = f.trace_id
WHERE t.created_at >= DATE('now', '-7 days')
GROUP BY DATE(t.created_at)
ORDER BY day DESC;
"""
df = pd.read_sql_query(eval_sql, conn)
print(df.to_string(index=False))


In [ ]:
# L6 add-on: A/B test by prompt_version
ab_sql = """
SELECT
    t.prompt_version,
    DATE(t.created_at) AS day,
    COUNT(*) AS total,
    AVG(CASE WHEN f.rating = 'thumbs_up' THEN 1.0 ELSE 0.0 END) AS satisfaction,
    AVG(t.latency_ms) AS avg_lat_ms
FROM traces t
LEFT JOIN feedback f ON t.trace_id = f.trace_id
WHERE t.created_at >= DATE('now', '-7 days')
GROUP BY t.prompt_version, DATE(t.created_at)
ORDER BY day DESC, prompt_version;
"""
df_ab = pd.read_sql_query(ab_sql, conn)
print("A/B test by prompt_version:")
print(df_ab.to_string(index=False))


In [ ]:
# L6 add-on: build the gold eval set from corrected_answer
gold_sql = """
SELECT
    t.query,
    t.response AS model_answer,
    f.corrected_answer AS human_answer,
    t.prompt_version
FROM traces t
JOIN feedback f ON t.trace_id = f.trace_id
WHERE f.corrected_answer IS NOT NULL
ORDER BY t.created_at DESC
LIMIT 5;
"""
df_gold = pd.read_sql_query(gold_sql, conn)
print("Gold eval set sample (rows where humans corrected the model):")
print(df_gold.to_string(index=False))
print()
print("-> This is your labeled data for offline eval. Replay new prompts against")
print("  this set BEFORE shipping. If satisfaction drops, do not ship.")


---

## ROUND 3 - Sr Tech Design: People-You-May-Know Feed with AI Summaries

**What they test:** Can you design a system that DOESN'T HALLUCINATE at Meta scale?

**The 3-layer anti-hallucination architecture:**
```
Layer 1: GRAPH (offline)
   friendships SCD2 -> Spark GraphX -> pymk_candidates

Layer 2: REASONS (offline, NO free LLM generation)
   candidate_reasons with TEMPLATE + evidence_json
   LLM only FILLS template with grounded evidence

Layer 3: SERVING (online)
   API -> merge offline candidates + Redis live mutuals + privacy check
   NO LLM at request time - reasons are pre-computed
```

**Why this design:**
- **SCD2 on friendships** = time travel. "Users who were friends 6 months ago" is a valid query for recency scoring.
- **Bounded fanout** = skip users with >5k friends. A user with 10k friends generates 10k squared = 100M writes per friend-add.
- **Grounded generation only** = the prompt MUST include `evidence_json` and the instruction "use ONLY provided evidence". No evidence -> output "unknown".
- **No LLM in serving path** = serving path is just a Redis lookup + template fill. LLM only runs offline for candidate_reasons pre-computation.


**L6 talking points:**
1. "Friendships SCD2 lets me answer 'who was this user's friend 6 months ago' - needed for recency-aware PYMK ranking."
2. "Bounded fanout: I skip events for users with >5000 friends. A celebrity with 50k friends would generate 2.5B writes per friend-add. That's not a system, that's a DDoS."
3. "Grounded generation prompt: 'Use ONLY facts present in evidence_json. If not present, output unknown.' No free LLM in serving path - that is how you get hallucinations at 2am."
4. "Reasons are pre-computed offline by the LLM using evidence_json. Serving path is just a Redis lookup + template substitution. P99 latency stays under 200ms."
5. "Privacy: I check the `blocks` table in the API gateway BEFORE returning. A user who blocked the candidate must never see them in PYMK, even if they share 50 mutuals."
6. "LLM-as-judge: nightly job samples 1000 reasons, checks if every claim is grounded in evidence_json. If >1% fail, alert + investigate the evidence pipeline."


---

## ROUND 4 - Full Stack: Build RAG Trace Reviewer Tool

**What they test:** Can you SHIP a working internal tool fast? They want SPEED + API design, not auth or full CRUD.

**The 3-end-point minimum viable tool:**
- `GET /traces?day=...&prompt_version=...&cursor=...&limit=50`
- `POST /feedback {trace_id, rating, corrected_answer}`
- `GET /traces/{trace_id}` for detail view

**Why cursor pagination, not OFFSET:**
- OFFSET 1000 LIMIT 50 = scan 1050 rows, throw away 1000. At 1B rows/day, this kills the DB.
- Cursor `WHERE trace_id < last_id ORDER BY created_at DESC LIMIT 50` = index seek, O(log n).
- Same UX for the user (infinite scroll), 1000x faster backend.


**L6 talking points while coding:**
1. "Cursor pagination, not OFFSET. OFFSET scans O(n) rows, cursor is an index seek."
2. "Filter by `prompt_version` from day 1 - you will need it for A/B analysis tomorrow."
3. "Optimistic UI on feedback - update the UI immediately, rollback if the server fails. Users hate waiting for a network round-trip on a thumbs up."
4. "Virtualized list (react-window) - 1k rows will tank React if I render naively. FixedSizeList only mounts visible rows."
5. "POST /feedback writes to a sync queue, not directly to the DB. The 30% feedback rate would saturate Postgres if synchronous. Async batch insert every 5 sec."
6. "Cap limit at 200. I have seen a teammate pass limit=10000 and bring down a staging cluster."


---

## ROUND 5 - Sr Ownership (Behavioral, 30m)

**What they test:** L6 != "I helped". L6 = "I owned end-to-end, defined the metric, and prevented recurrence."

**The L6 STAR upgrade:**
```
L4 STAR:  S/T/A/R as described in job ad
L5 STAR:  S/T/A/R with my specific actions
L6 STAR:  S/T/A/R + METRIC + PREVENTION
                              ^           ^
                          quantified    runbook/SLO
                          outcome       you left behind
```

### Q1: Owned a broken pipeline no one wanted

**L6 ANSWER:**
> **S:** RAG p95 latency went from 400ms to 2.5s after we upgraded the embedding model.
>
> **T:** No owner. On-call was getting paged at 2am. Leadership wanted answers.
>
> **A:** I owned it end-to-end:
> 1. Rolled back the index to v1 to stop the bleeding - 10 min MTTR
> 2. Found root cause: smaller batch size caused Qdrant HNSW graph fragmentation
> 3. Built a dashboard: `embedding_drift_by_model_version` + `p95_by_version`
> 4. Wrote a runbook: how to roll back, how to detect drift, on-call playbook
> 5. Added an alert: p95 > 1s for 5 min pages on-call
>
> **R:** Latency back to 380ms. New SLO: p95 < 500ms. My dashboard is now the team's standard. Zero similar incidents in 6 months.

**L6 keywords:** owned, rolled back, root cause, dashboard, runbook, SLO, prevented recurrence.

### Q2: DE vs ML team conflict on data quality

**L6 ANSWER:**
> "I proposed a **data contract**:
> - ML team defines the **gold dataset schema** (what 'good data' looks like)
> - DE team guarantees **freshness SLA** (data lands within 5 min of source)
> - DE team guarantees **schema SLA** (no breaking changes without notice)
>
> I wrote the contract as JSON, added **Great Expectations checks** in the pipeline that fail the DAG if contract breaks. Both teams now have measurable SLAs. Conflict resolved with metrics, not opinions."

### Q3: Vague problem "Make RAG better"

**L6 ANSWER:**
> "First, I defined 'better' - I refused to optimize without a metric.
>
> 1. **Defined the metric:** satisfaction = thumbs_up / total, weighted by latency
> 2. **Built the eval set:** mined `feedback.corrected_answer` from prod, got 2k labeled examples
> 3. **Hypothesized:** chunk size 512 + reranker > current 256 no reranker
> 4. **Ran A/B:** 12% satisfaction lift, +50ms latency (acceptable)
> 5. **Shipped it** with monitoring and rollback plan
>
> The key was refusing to start work without a measurable definition of success."

**L6 keywords:** defined metric, eval set, A/B test, shipped, monitored.


---

## VECTOR DB DEMO (SQLite + sqlite-vec = pgvector equivalent)

**What this proves:** The exact same RAG schema and KNN queries run on pgvector in production. We use sqlite-vec here for portability - same SQL syntax for `float[N]` columns, same KNN semantics with `vec_distance_cosine` and `WHERE embedding MATCH ?`.

**What the demo does (4 parts):**
1. **KNN vector search** - 'When was the company started?' -> returns Meta history chunks in correct order
2. **GDPR tombstone** - user_dave deletes account, async purge cascades through chunks + embeddings
3. **Dual-write re-embed** - new model version writes to a shadow vec0 index, prod keeps v1
4. **Analytics SQL** - satisfaction + latency by prompt_version (A/B test)

**Why this matters for the interview:** You can say 'I built this schema, KNN retrieves Top 3 in this order, GDPR works, re-embed is dual-write, A/B SQL gives us the eval signal.' That is the full L6 data model in <300 lines.


In [ ]:
"""
Vector DB demo - SQLite + sqlite-vec (drop-in for pgvector)
Same SQL concepts as pgvector: CREATE TABLE ... embedding float[N], KNN queries with distance.
"""
import sqlite3
import sqlite_vec
import numpy as np
import json
import uuid
from datetime import datetime

# ─── Connect and enable the vector extension ───────────────────────
conn = sqlite3.connect(':memory:')
conn.enable_load_extension(True)
sqlite_vec.load(conn)
conn.enable_load_extension(False)

print(f"sqlite-vec version: {sqlite_vec.__version__}")
print("Vector extension loaded OK\n")

# ─── Schema (the L6 RAG model) ─────────────────────────────────────
conn.executescript("""
-- 1. documents: source of truth with GDPR tombstone
CREATE TABLE documents (
    doc_id          TEXT PRIMARY KEY,
    source          TEXT,
    uri             TEXT,
    owner_id        TEXT,
    chunk_strategy  TEXT,
    ingested_at     TEXT,
    is_deleted      INTEGER DEFAULT 0,  -- tombstone, NOT hard delete
    hash            TEXT
);

-- 2. chunks: the actual data model
CREATE TABLE chunks (
    chunk_id               TEXT PRIMARY KEY,
    doc_id                 TEXT REFERENCES documents(doc_id),
    chunk_index            INTEGER,
    text                   TEXT,
    token_count            INTEGER,
    embedding_model_version TEXT,
    chunk_hash             TEXT,
    created_at             TEXT,
    UNIQUE (doc_id, chunk_hash, embedding_model_version)
);

-- 3. embeddings: separate table (vector(1536) is 6KB/row)
CREATE TABLE chunks_vec (
    chunk_id      TEXT PRIMARY KEY,
    embedding     float[8],          -- using 8-dim for demo speed
    model_version TEXT,
    created_at    TEXT
);

-- 4. prompts: versioned like code
CREATE TABLE prompts (
    prompt_id       TEXT PRIMARY KEY,
    prompt_template TEXT,
    version         INTEGER,
    created_at      TEXT
);

-- 5. traces: fact table (would be PARTITIONED in prod)
CREATE TABLE traces (
    trace_id            TEXT PRIMARY KEY,
    user_id             TEXT,
    query               TEXT,
    retrieved_chunk_ids TEXT,   -- JSON array
    prompt_version      INTEGER,
    response            TEXT,
    latency_ms          INTEGER,
    cost_tokens         INTEGER,
    created_at          TEXT
);

-- 6. feedback: gold dataset
CREATE TABLE feedback (
    feedback_id      TEXT PRIMARY KEY,
    trace_id         TEXT,
    rating           TEXT CHECK (rating IN ('thumbs_up', 'thumbs_down')),
    corrected_answer TEXT,
    created_at       TEXT
);
""")
print("Schema created: 6 tables (documents, chunks, chunks_vec, prompts, traces, feedback)\n")

# ─── Seed sample data ──────────────────────────────────────────────
now = datetime.utcnow().isoformat()

# 4 sample documents
docs = [
    ("doc_001", "sharepoint", "https://meta.sharepoint.com/meta-history", "user_alice", "langchain_512", "meta_history_overview"),
    ("doc_002", "confluence", "https://meta.atlassian.net/rag-design",  "user_bob",   "semantic_v1",   "rag_architecture_doc"),
    ("doc_003", "s3",         "s3://meta-internal/eval-results-2026",  "user_carol", "langchain_512", "eval_results_q3"),
    ("doc_004", "sharepoint", "https://meta.sharepoint.com/llm-traces", "user_dave",  "langchain_512", "llm_tracing_guide"),
]

for doc_id, source, uri, owner, strategy, hash_val in docs:
    conn.execute(
        "INSERT INTO documents VALUES (?,?,?,?,?,?,?,?)",
        (doc_id, source, uri, owner, strategy, now, 0, hash_val)
    )
print(f"Inserted {len(docs)} documents\n")

# Sample chunks - each doc has 2-3 chunks
chunk_seeds = [
    # (chunk_id, doc_id, idx, text, tokens, model_ver, chunk_hash)
    ("chk_001", "doc_001", 0, "Meta was founded in 2004 by Mark Zuckerberg at Harvard.",            18, "ada-002", "h001"),
    ("chk_002", "doc_001", 1, "The company went public in 2012 with the largest tech IPO at the time.", 22, "ada-002", "h002"),
    ("chk_003", "doc_001", 2, "Meta acquired Instagram in 2012 and WhatsApp in 2014.",                14, "ada-002", "h003"),
    ("chk_004", "doc_002", 0, "RAG combines retrieval from a vector DB with LLM generation.",         16, "ada-002", "h004"),
    ("chk_005", "doc_002", 1, "Hybrid search uses both BM25 keyword search and vector similarity.",    14, "ada-002", "h005"),
    ("chk_006", "doc_002", 2, "Cross-encoder rerankers improve precision at the cost of latency.",    13, "ada-002", "h006"),
    ("chk_007", "doc_003", 0, "Eval dataset size correlates with offline metric reliability.",       12, "ada-002", "h007"),
    ("chk_008", "doc_003", 1, "LLM-as-judge achieves 85% agreement with human raters on factual QA.", 15, "ada-002", "h008"),
    ("chk_009", "doc_004", 0, "Every LLM call should be traced with prompt version, latency, and cost.", 16, "ada-002", "h009"),
    ("chk_010", "doc_004", 1, "Traces are partitioned by day. Drop partitions after 30 days.",      12, "ada-002", "h010"),
]

for chunk_id, doc_id, idx, text, tokens, model_ver, c_hash in chunk_seeds:
    conn.execute(
        "INSERT INTO chunks VALUES (?,?,?,?,?,?,?,?)",
        (chunk_id, doc_id, idx, text, tokens, model_ver, c_hash, now)
    )
print(f"Inserted {len(chunk_seeds)} chunks\n")

# ─── Generate fake embeddings (8-dim for demo) ─────────────────────
np.random.seed(42)

# Make semantically related chunks have similar embeddings
# docs about Meta history -> one cluster
# docs about RAG -> another cluster
# docs about eval -> third
# docs about tracing -> fourth
embeddings_map = {
    "chk_001": np.array([0.9, 0.1, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "chk_002": np.array([0.85, 0.15, 0.05, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "chk_003": np.array([0.8, 0.2, 0.1, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "chk_004": np.array([0.0, 0.0, 0.9, 0.1, 0.0, 0.0, 0.1, 0.0]),
    "chk_005": np.array([0.0, 0.0, 0.85, 0.15, 0.0, 0.0, 0.1, 0.05]),
    "chk_006": np.array([0.0, 0.0, 0.8, 0.2, 0.0, 0.0, 0.1, 0.05]),
    "chk_007": np.array([0.0, 0.1, 0.0, 0.0, 0.9, 0.1, 0.0, 0.0]),
    "chk_008": np.array([0.0, 0.1, 0.0, 0.0, 0.85, 0.15, 0.0, 0.0]),
    "chk_009": np.array([0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.9, 0.1]),
    "chk_010": np.array([0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.85, 0.15]),
}

# Create a vec0 VIRTUAL TABLE for true KNN (the production-style pattern)
conn.execute("CREATE VIRTUAL TABLE chunks_vec_idx USING vec0(embedding float[8], chunk_id text, model_version text)")

# Insert into both: the regular table (analytics) AND the vec0 index (KNN)
for chunk_id, vec in embeddings_map.items():
    # Regular table for metadata queries
    conn.execute(
        "INSERT INTO chunks_vec VALUES (?, ?, ?, ?)",
        (chunk_id, sqlite_vec.serialize_float32(vec.tolist()), "ada-002", now)
    )
    # vec0 virtual table for fast KNN
    conn.execute(
        "INSERT INTO chunks_vec_idx(rowid, embedding, chunk_id, model_version) VALUES (?, ?, ?, ?)",
        (hash(chunk_id) & 0x7FFFFFFF, sqlite_vec.serialize_float32(vec.tolist()), chunk_id, "ada-002")
    )
print(f"Inserted {len(embeddings_map)} 8-dim embeddings into chunks_vec + chunks_vec_idx\n")

# ─── Versioned prompts ────────────────────────────────────────────
prompts = [
    ("prm_001", "You are a helpful assistant. Use the context to answer: {context}\nQ: {query}", 1, now),
    ("prm_002", "You are a Meta engineer. Ground answers in the docs. If unknown say 'unknown'.\n{context}\nQ: {query}", 2, now),
    ("prm_003", "Cite your sources. If the docs don't say, say 'I don't have that info'.\n{context}\nQ: {query}", 3, now),
]
for p_id, template, ver, ts in prompts:
    conn.execute("INSERT INTO prompts VALUES (?,?,?,?)", (p_id, template, ver, ts))
print(f"Inserted {len(prompts)} prompt versions (v1, v2, v3)\n")

# ─── Sample traces (3 example queries) ───────────────────────────
traces = [
    ("trc_001", "user_alice", "When was Meta founded?",          json.dumps(["chk_001","chk_002"]), 1, "Meta was founded in 2004.",         420, 850,  now),
    ("trc_002", "user_bob",   "How does RAG work?",              json.dumps(["chk_004","chk_005"]), 2, "RAG combines retrieval and gen.",   480, 920,  now),
    ("trc_003", "user_carol", "Is LLM-as-judge reliable?",       json.dumps(["chk_008"]),           2, "Yes, 85% agreement with humans.",  390, 750,  now),
]
for trc in traces:
    conn.execute("INSERT INTO traces VALUES (?,?,?,?,?,?,?,?,?)", trc)
print(f"Inserted {len(traces)} traces\n")

# ─── Feedback (gold set) ──────────────────────────────────────────
feedback = [
    ("fdb_001", "trc_001", "thumbs_up",   None,                now),
    ("fdb_002", "trc_002", "thumbs_down", "RAG = retrieval + generation step.",  now),
    ("fdb_003", "trc_003", "thumbs_up",   None,                now),
]
for fb in feedback:
    conn.execute("INSERT INTO feedback VALUES (?,?,?,?,?)", fb)
print(f"Inserted {len(feedback)} feedback rows\n")

conn.commit()

# ═══════════════════════════════════════════════════════════════════
# DEMO 1: Vector similarity search (the actual RAG retrieval)
# ═══════════════════════════════════════════════════════════════════
print("=" * 70)
print("DEMO 1: Vector similarity search")
print("=" * 70)

# Query: "When was the company started?" -> should match Meta history chunks
query = np.array([0.88, 0.12, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0])  # close to chk_001/002/003
query_serialized = sqlite_vec.serialize_float32(query.tolist())

# KNN via vec0 virtual table (production-style: this is the pgvector equivalent)
results = conn.execute("""
    SELECT
        chunk_id,
        distance
    FROM chunks_vec_idx
    WHERE embedding MATCH ?
    ORDER BY distance
    LIMIT 3
""", (query_serialized,)).fetchall()

print(f"\nQuery: 'When was the company started?'\n")
print("Top 3 chunks by cosine distance (KNN):")
for chunk_id, dist in results:
    row = conn.execute("SELECT text FROM chunks WHERE chunk_id = ?", (chunk_id,)).fetchone()
    print(f"  [{dist:.4f}] {chunk_id}: {row[0]}")

# ═══════════════════════════════════════════════════════════════════
# DEMO 2: GDPR tombstone + vector delete
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("DEMO 2: GDPR tombstone (user_dave deletes their account)")
print("=" * 70)

# Tombstone the document
conn.execute("UPDATE documents SET is_deleted = 1 WHERE owner_id = 'user_dave'")
print("Tombstoned doc_004 (owner user_dave)")

# Show what would happen in the async purge job
print("\nAsync purge job would now:")
print("  1. Find is_deleted = 1 documents")
print("  2. DELETE FROM chunks WHERE doc_id IN (...)")
print("  3. DELETE FROM chunks_vec WHERE chunk_id IN (...)")
print("  4. Keep traces for 30 days for abuse investigation")

# Simulate the purge
deleted = conn.execute("""
    SELECT COUNT(*) FROM chunks c
    JOIN documents d ON c.doc_id = d.doc_id
    WHERE d.is_deleted = 1
""").fetchone()[0]
print(f"\nWould purge {deleted} chunks from vector DB")

# ═══════════════════════════════════════════════════════════════════
# DEMO 3: Re-embedding with new model (dual-write pattern)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("DEMO 3: Re-embed with new model version (dual-write)")
print("=" * 70)

# New model: text-embedding-3-large (different distribution)
np.random.seed(99)
new_embeddings = {
    "chk_001": np.array([0.95, 0.05, 0.0, 0.0, 0.0, 0.05, 0.0, 0.0]),
    "chk_002": np.array([0.92, 0.08, 0.0, 0.0, 0.0, 0.05, 0.0, 0.0]),
}
# Write to a SHADOW vec0 index (embeddings_v2) - the dual-write pattern
conn.execute("""
    CREATE VIRTUAL TABLE chunks_vec_v2 USING vec0(embedding float[8], chunk_id text, model_version text)
""")

for chunk_id, vec in new_embeddings.items():
    conn.execute(
        "INSERT INTO chunks_vec_v2(rowid, embedding, chunk_id, model_version) VALUES (?, ?, ?, ?)",
        (hash(chunk_id + "_v2") & 0x7FFFFFFF, sqlite_vec.serialize_float32(vec.tolist()), chunk_id, "text-embedding-3-large")
    )
print(f"Dual-wrote {len(new_embeddings)} chunks to chunks_vec_v2 (shadow index)")
print("In prod: serve prod from v1, dual-read 1% from v2 for 7 days, then cutover")

# ═══════════════════════════════════════════════════════════════════
# DEMO 4: Analytics query (the SQL eval from notebook cell 5)
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("DEMO 4: RAG quality analytics")
print("=" * 70)

import pandas as pd
df = pd.read_sql_query("""
    SELECT
        t.prompt_version,
        COUNT(*) AS total,
        AVG(CASE WHEN f.rating = 'thumbs_up' THEN 1.0 ELSE 0.0 END) AS satisfaction,
        AVG(t.latency_ms) AS avg_latency
    FROM traces t
    LEFT JOIN feedback f ON t.trace_id = f.trace_id
    GROUP BY t.prompt_version
    ORDER BY t.prompt_version
""", conn)
print("\nA/B test by prompt_version:")
print(df.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
counts = {
    "documents":     conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0],
    "chunks":        conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0],
    "embeddings":    conn.execute("SELECT COUNT(*) FROM chunks_vec").fetchone()[0],
    "prompts":       conn.execute("SELECT COUNT(*) FROM prompts").fetchone()[0],
    "traces":        conn.execute("SELECT COUNT(*) FROM traces").fetchone()[0],
    "feedback":      conn.execute("SELECT COUNT(*) FROM feedback").fetchone()[0],
}
for table, n in counts.items():
    print(f"  {table:15s}: {n} rows")
print(f"\nVector DB working. Same SQL would run on pgvector in prod.")


---

## How to Use This Notebook

1. **Run all cells** (Cell > Run All) - verifies MMR algorithm, SQL queries, A/B test all work
2. **Modify the MMR lambda** - try lambda=0.3 to see how diversity changes the output
3. **Modify the SQL** - try GROUP BY user_id instead of day to see cohort analysis
4. **Read META_L6_STUDY_GUIDE.md** for the theory and L6 answer scripts
5. **Say "start round 2"** to begin live mock mode with me as interviewer

## Files in This Folder
- `meta_l6_onsite_mock.ipynb` - this notebook (runnable)
- `META_L6_STUDY_GUIDE.md` - theory + scripts
- `cells/` - source files for each cell
- `build_notebook.py` - script to rebuild this notebook from cells/
